Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

Load the data

In [23]:
from src.dataloader import DataLoader
from src.config import load_config

config = load_config('../configs/base.yaml')

dataloader = DataLoader(
    poly_path=config['paths']['data']['poly_path'],
    site_path=config['paths']['data']['site_path'],
    rast_path=config['paths']['data']['rast_path'],
    label_path=config['paths']['data']['label_path'],
    data_path=config['paths']['data']['data_path'],
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    download=config['data']['download'],
    load=config['data']['load'],
    normalize=config['data']['normalize']
)

Create a temporal graph with node features and targets

In [32]:
from torch_geometric_temporal.signal import temporal_signal_split

config = load_config('../configs/base.yaml')
data = dataloader.get_graph(k=config['graph']['k'],
                            r=config['graph']['r'],
                            normalize=config['data']['normalize'],
                            lag=config['graph']['lag'])

Transform the graph into a graph with static features with the signature

In [65]:
from src.signature import SignatureFeatures

# Configure the signature transform as needed
config = load_config('../configs/base.yaml')
sig_transform = SignatureFeatures(
    sig_depth=config['sig']['depth'],         # or your desired depth
    normalize=config['sig']['normalize'],      # whether to normalize the signature features
    log_signature=config['sig']['log_signature'], # use log-signature or not
    time_augment=config['sig']['time_augment'],  # add time as a feature or not
    lead_lag=config['sig']['lead_lag']      # use lead-lag augmentation or not
)

# Apply the transform to your temporal graph data
static_graph = sig_transform(data)

Train a GCN classifier to predict labels

In [88]:
import torch
from signatory import signature_channels
from src.graph import NodeSplitMask
from src.model import ClassifierGCN


hyper = load_config('../configs/hyper.yaml')
node_features = signature_channels(3, config['sig']['depth'])

split_transform = NodeSplitMask(train_ratio=hyper['split']['train_ratio'],
                                val_ratio=hyper['split']['val_ratio'],
                                test_ratio=hyper['split']['test_ratio'],
                                seed=hyper['split']['seed'])
static_graph = split_transform(static_graph)


model = ClassifierGCN(node_features,
                      hidden_features=hyper['hidden_features'],
                      num_classes=hyper['num_classes'],
                      filter_size=hyper['filter_size'])

optimizer = torch.optim.Adam(model.parameters(),
                             lr=hyper['lr'],
                             weight_decay=hyper['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()

# Training loop
epochs = hyper
for epoch in range(int(hyper['num_epochs'])):
    model.train()
    optimizer.zero_grad()
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    # Only use labeled nodes for training
    loss = criterion(out[static_graph.train_mask], static_graph.y[static_graph.train_mask].long())
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        pred = out.argmax(dim=1)
        train_acc = (pred[static_graph.train_mask] == static_graph.y[static_graph.train_mask]).float().mean().item()
        val_acc = (pred[static_graph.val_mask] == static_graph.y[static_graph.val_mask]).float().mean().item() if static_graph.val_mask.sum() > 0 else float('nan')
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f}")

# Test evaluation
model.eval()
with torch.no_grad():
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    pred = out.argmax(dim=1)
    test_acc = (pred[static_graph.test_mask] == static_graph.y[static_graph.test_mask]).float().mean().item() if static_graph.test_mask.sum() > 0 else float('nan')
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 000 | Loss: 0.6143 | Train Acc: 0.7143
Epoch 010 | Loss: 0.5697 | Train Acc: 0.7143
Epoch 020 | Loss: 0.5297 | Train Acc: 0.7381
Epoch 030 | Loss: 0.4942 | Train Acc: 0.7381
Epoch 040 | Loss: 0.4627 | Train Acc: 0.8333
Epoch 050 | Loss: 0.4346 | Train Acc: 0.8810
Epoch 060 | Loss: 0.4095 | Train Acc: 0.8810
Epoch 070 | Loss: 0.3872 | Train Acc: 0.8810
Epoch 080 | Loss: 0.3673 | Train Acc: 0.9048
Epoch 090 | Loss: 0.3492 | Train Acc: 0.9048
Epoch 100 | Loss: 0.3327 | Train Acc: 0.9048
Epoch 110 | Loss: 0.3174 | Train Acc: 0.9048
Epoch 120 | Loss: 0.3030 | Train Acc: 0.9048
Epoch 130 | Loss: 0.2895 | Train Acc: 0.9286
Epoch 140 | Loss: 0.2769 | Train Acc: 0.9286
Epoch 150 | Loss: 0.2652 | Train Acc: 0.9286
Epoch 160 | Loss: 0.2542 | Train Acc: 0.9286
Epoch 170 | Loss: 0.2438 | Train Acc: 0.9524
Epoch 180 | Loss: 0.2340 | Train Acc: 0.9524
Epoch 190 | Loss: 0.2247 | Train Acc: 0.9524
Epoch 200 | Loss: 0.2158 | Train Acc: 0.9524
Epoch 210 | Loss: 0.2073 | Train Acc: 0.9524
Epoch 220 

In [89]:
pred[static_graph.test_mask]

tensor([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1,
        0, 0, 1, 1])

In [90]:
static_graph.y[static_graph.test_mask]

tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])